# ❄️ Building an End to End Machine Learning Workflow in Snowflake ❄️

In this Notebook ([on Container Runtime](https://docs.snowflake.com/developer-guide/snowflake-ml/notebooks-on-spcs)), we will develop a machine learning model that accurately predicts the "mortgage response" (e.g., loan approval, offer acceptance) based on borrower characteristics and loan details.

**Why is this important?**

- `Risk Management:` Lenders can better assess the risk of loan default.
- `Operational Efficiency:` Automating parts of the approval process.
- `Targeted Marketing:` Identifying potential borrowers more effectively.
- `Improved Customer Experience:` Streamlining the loan process.

We will showcase all the typical steps in a machine learning pipeline using native capabilities in Snowflake through this use case:

### 1. `FEATURE ENGINEERING:` Use [Snowflake Feature Store](https://docs.snowflake.com/en/developer-guide/snowflake-ml/feature-store/overview) to track engineered features
- Store feature defintions in a feature store for reproducible computation of ML features
      
### 2. `MODEL TRAINING:` Train models using OSS XGBoost and Snowflake ML APIs
- Baseline OSS XGboost
- XGBoost with optimal hyperparameters identified via [Snowflake ML Parallel Hyperparameter Optimization](https://docs.snowflake.com/en/developer-guide/snowflake-ml/container-hpo)

### 3. `MODEL LOGGING, INFERENCE, & EXPLAINABILITY:` Register models in [Snowflake Model Registry](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)
- Explore model registry capabilities such as **metadata tracking, inference, and explainability**
- Compare model metrics on train/test set to identify any issues of model performance or overfitting
- Tag the best performing model version as 'default' version

### 4. `ML OBSERVABILITY:` Set up [Model Monitors](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/model-observability) to track 1 year of predicted and actual loan repayments
- **Compute performance metrics** such a F1, Precision, Recall
- **Inspect model drift** (i.e. how much has the average predicted repayment rate changed day-to-day)
- **Compare models** side-by-side to understand which model should be used in production
- Identify and understand **data issues**

### 5. `ML LINEAGE:` Track [data and model lineage](https://docs.snowflake.com/en/user-guide/ui-snowsight-lineage#ml-lineage) throughout
- View and understand
  - The **origin of the data** used for computed features
  - The **data used** for model training
  - The **available model versions** being monitored

### 6. `MODEL DEPLOYMENT`
- [Deploy model to Snowpark Container Services](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/container)

Let's set a version (must be a string) that will be used through all the artifacts that will be created for our project.

In [ ]:
VERSION_NUM = 'V0'

Now, let's import all the libraries we need.

In [ ]:
# Built-in
import math
import pickle
from datetime import datetime

# Third-party
import pandas as pd
import numpy as np
import shap
import sklearn
import streamlit as st
from xgboost import XGBClassifier

# Snowflake ML
from snowflake.ml.registry import Registry
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode
from snowflake.ml.runtime_cluster import scale_cluster, get_nodes

# Snowpark Core
from snowflake.snowpark import DataFrame, Window
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import (
    col,
    to_timestamp,
    min,
    max,
    month,
    dayofmonth,
    dayofweek,
    dayofyear,
    avg,
    date_add,
    sql_expr,
)
from snowflake.snowpark.types import IntegerType

import logging
logging.getLogger().setLevel(logging.INFO)

# Initialize session
session = get_active_session()
session

In [ ]:
# Use schema DEFAULT_SCHEMA to create new Snowflake objects in this project
session.use_schema('DEFAULT_SCHEMA')

We can read the data that's already been pre-loaded into your account as a Snowflake [Snowpark dataframe](https://docs.snowflake.com/en/developer-guide/snowpark/python/working-with-dataframes).

In [ ]:
df = session.table("MORTGAGE_LENDING_DEMO_DATA")
df.show()

# Feature Engineering with Snowpark APIs

We will create a number of features within `create_mortgage_features()`:
- Timestamp features (i.e. month, day of year, day of week)
- Income and Loan features
- County-level income stats
- High income flag
- Time-based rolling average.

All the features are created using Snowpark DF operations and functions, which you can find the API reference documentation for [here](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/1.2.0/index). 

In [ ]:
def create_mortgage_features(df):
    # Step 1: Timestamp features (Per-row features)
    # Get current date and time
    current_time = datetime.now()
    df_max_time = datetime.strptime(str(df.select(max("TS")).collect()[0][0]), "%Y-%m-%d %H:%M:%S.%f")
    
    # Find delta between latest existing timestamp and today's date
    timedelta = current_time- df_max_time

    df = df.with_columns(
        ["TIMESTAMP", "MONTH", "DAY_OF_YEAR", "DOTW"],
        [
            date_add(to_timestamp("TS"), timedelta.days-1),
            month("TIMESTAMP"),
            dayofyear("TIMESTAMP"),
            dayofweek("TIMESTAMP")
        ]
    )
    
    # Step 2: Income and loan features (Per-row features)
    df = df.with_columns(
        ["LOAN_AMOUNT", "INCOME", "INCOME_LOAN_RATIO"],
        [
            col("LOAN_AMOUNT_000s")*1000,
            col("APPLICANT_INCOME_000s")*1000,
            col("INCOME")/col("LOAN_AMOUNT")
        ]
    )
    
    # Step 3: County-level income stats (Per-group features)
    county_income_df = df.group_by(["COUNTY_NAME"]).agg(
        avg("INCOME").alias("AVG_COUNTY_INCOME")
    )
    # Join back to the original dataframe
    df = df.join(county_income_df, "COUNTY_NAME")
    
    # Step 4: Add high income flag
    df = df.with_column(
        "HIGH_INCOME_FLAG", 
        (col("INCOME") > col("AVG_COUNTY_INCOME")).astype(IntegerType())
    )
    
    # Step 5: Time-based rolling average
    df = df.with_column(
        "AVG_THIRTY_DAY_LOAN_AMOUNT",
        sql_expr("""
            AVG(LOAN_AMOUNT) OVER (
                PARTITION BY COUNTY_NAME 
                ORDER BY TIMESTAMP 
                RANGE BETWEEN INTERVAL '30 DAYS' PRECEDING AND CURRENT ROW
            )
        """)
    )

    return df
    
df = create_mortgage_features(df)

feature_df = df.select(
        ["LOAN_ID", "TIMESTAMP", "MONTH", "DAY_OF_YEAR", "DOTW", 
         "LOAN_AMOUNT", "INCOME", "INCOME_LOAN_RATIO", 
         "AVG_COUNTY_INCOME", "HIGH_INCOME_FLAG", 
         "AVG_THIRTY_DAY_LOAN_AMOUNT"]
    )

Let's take a look at the features we just created:

In [ ]:
feature_df.limit(5)

We can even see what the actual SQL execution plan in under the hood from defining all the feature logic above using Snowpark.

In [ ]:
feature_df.explain()

## Now, we can create a [Snowflake Feature Store](https://docs.snowflake.com/en/developer-guide/snowflake-ml/feature-store/overview).

We'll go ahead and use our session's current Snowflake database, schema, and warehouse to create it for the purpose of this demo.

In [ ]:
fs = FeatureStore(
    session=session, 
    database=session.get_current_database(), 
    name=session.get_current_schema(), 
    default_warehouse=session.get_current_warehouse(),
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

A feature store contains feature views. 

A **feature view** encapsulates a Python or SQL pipeline for transforming raw data into one or more related features. 

Feature views are organized in the feature store according to the **entities** to which they apply. An **entity** is a higher-level abstraction that represents the subject matter of a feature. 

So, let's proceed to create a new entity now. Since we are creating features at the loan level, we will create an entity for loans. Entities are used to look up and join features.

In [ ]:
# First try to retrieve an existing entity definition
# If not, define a new one and register
try:
    # Retrieve existing entity
    loan_id_entity = fs.get_entity('LOAN_ENTITY') 
    print('Retrieved existing entity')
except:
    # Define new entity
    loan_id_entity = Entity(
        name = "LOAN_ENTITY",
        join_keys = ["LOAN_ID"],
        desc = "Features defined on a per loan level")
    # Register
    fs.register_entity(loan_id_entity)
    print("Registered new entity")

In [ ]:
fs.list_entities()

Now, we can create a [Feature View](https://docs.snowflake.com/en/developer-guide/snowflake-ml/feature-store/feature-views) based on the feature engineering logic we created above and applied to the data we read into a Snowpark DF, which is encapusulated into `feature_df`.

In [ ]:
# Define and register feature view
loan_fv = FeatureView(
    name="Mortgage_Feature_View",
    entities=[loan_id_entity],
    feature_df=feature_df,
    timestamp_col="TIMESTAMP",
    refresh_freq="1 day",
    refresh_mode="INCREMENTAL")

# Add feature level descriptions
loan_fv = loan_fv.attach_feature_desc(
    {
        "MONTH": "Month of loan",
        "DAY_OF_YEAR": "Day of calendar year of loan",
        "DOTW": "Day of the week of loan",
        "LOAN_AMOUNT": "Loan amount in $USD",
        "INCOME": "Household income in $USD",
        "INCOME_LOAN_RATIO": "Ratio of LOAN_AMOUNT/INCOME",
        "AVG_COUNTY_INCOME": "Average household income aggregated at county level",
        "HIGH_INCOME_FLAG": "Binary flag to indicate whether household income is higher than MEDIAN_COUNTY_INCOME",
        "AVG_THIRTY_DAY_LOAN_AMOUNT": "Rolling 30 day average of LOAN_AMOUNT"
    }
)

loan_fv = fs.register_feature_view(loan_fv, version=VERSION_NUM, overwrite=True)

Let's take a look at the existing feature views we have in our Feature Store through the Python API.

In [ ]:
fs.list_feature_views()

You can also inspect your feature view in the Feature Store UI link generated in the next cell. 

You can also click on the `Lineage` tab in the Feature View to look at the lineage of these features. 

In [ ]:
# Create link to feature store UI to inspect newly created feature view!
org_name = session.sql('SELECT CURRENT_ORGANIZATION_NAME()').collect()[0][0]
account_name = session.sql('SELECT CURRENT_ACCOUNT_NAME()').collect()[0][0]
db_name = session.sql('SELECT CURRENT_DATABASE()').collect()[0][0]
schema_name = session.sql('SELECT CURRENT_SCHEMA()').collect()[0][0]

st.write(f'https://app.snowflake.com/{org_name}/{account_name}/#/features/database/{db_name}/store/{schema_name}')

### Retrieve a [Dataset from the feature view for model training](https://docs.snowflake.com/en/developer-guide/snowflake-ml/feature-store/modeling#generating-snowflake-datasets-for-training)

We can now generate a [Snowflake Dataset](https://docs.snowflake.com/en/developer-guide/snowflake-ml/dataset), which are immutable, file-based objects that exist within your Snowpark session. 

They can be written to persistent Snowflake objects as needed.

First, we create a "spine dataframe" which will contain the rows and entity IDs that we want to include in our training data. In this example, we are selecting all the rows in our source data. 

In [ ]:
# Create a spine dataframe to select the desired rows for training. 
# In this case we are using all the rows 
spine_df = df.select("LOAN_ID", "TIMESTAMP", "LOAN_PURPOSE_NAME","MORTGAGERESPONSE") #only need the features used to fetch rest of feature view
spine_df.show()

The `generate_dataset()` API will fill this spine_df with the correct feature values from the Feature View.

In [ ]:
ds = fs.generate_dataset(
    name=f"MORTGAGE_DATASET_EXTENDED_FEATURES_{VERSION_NUM}",
    # only need the features used to fetch rest of feature view
    spine_df=df.select("LOAN_ID", "TIMESTAMP", "LOAN_PURPOSE_NAME","MORTGAGERESPONSE"), 
    features=[loan_fv],
    spine_timestamp_col="TIMESTAMP",
    spine_label_cols=["MORTGAGERESPONSE"]
)

In [ ]:
ds_sp = ds.read.to_snowpark_dataframe()
ds_sp.show(5)

Next, we will use [Snowflake ML distributed preprocessors such as OneHotEncoder](https://docs.snowflake.com/en/developer-guide/snowflake-ml/modeling#preprocessing) which are implementations of sklearn preprocessors and run in parallel on the Warehouse. Alternatively you can use OSS sklearn preprocessors directly on the Container Runtime.

In [ ]:
import snowflake.ml.modeling.preprocessing as snowml
from snowflake.snowpark.types import StringType

OHE_COLS = ds_sp.select([col.name for col in ds_sp.schema if col.datatype ==StringType()]).columns
OHE_POST_COLS = [i+"_OHE" for i in OHE_COLS]

# Encode categoricals to numeric columns
snowml_ohe = snowml.OneHotEncoder(input_cols=OHE_COLS, output_cols = OHE_COLS, drop_input_cols=True)
ds_sp_ohe = snowml_ohe.fit(ds_sp).transform(ds_sp)

# Rename columns to avoid double nested quotes and white space chars
rename_dict = {}
for i in ds_sp_ohe.columns:
    if '"' in i:
        rename_dict[i] = i.replace('"','').replace(' ', '_')

ds_sp_ohe = ds_sp_ohe.rename(rename_dict)
ds_sp_ohe.columns

Now, we're ready to split the processed data into train/test sets, fill any null values, and convert the data to pandas to use OSS XGBoost.

In [ ]:
train, test = ds_sp_ohe.random_split(weights=[0.70, 0.30], seed=0)

In [ ]:
cols = list(ds_sp_ohe.columns)
cols.remove("TIMESTAMP")

train = train.fillna(0, subset=cols)
test = test.fillna(0, subset=cols)

In [ ]:
train_pd = train.to_pandas()
test_pd = test.to_pandas()

# Model Training, Model Logging, Inference, & Explainability

## Configure the base OSS XGBoost model & fit the model

Let's define and train a model using the OSS `xgboost` library. Note in our imports at the top: `from xgboost import XGBClassifier`

In [ ]:
# Define model config
xgb_base = XGBClassifier(
    max_depth=50,
    n_estimators=3,
    learning_rate = 0.75,
    booster = 'gbtree')

In [ ]:
# Split train data into X, y
X_train_pd = train_pd.drop(["TIMESTAMP", "LOAN_ID", "MORTGAGERESPONSE"],axis=1) #remove
y_train_pd = train_pd.MORTGAGERESPONSE

# Train model
xgb_base.fit(X_train_pd,y_train_pd)

Let's measure the baseline performance of our first version.

Note that here, we simply use OSS methods from `scikit-learn`.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
train_preds_base = xgb_base.predict(X_train_pd)

f1_base_train = round(f1_score(y_train_pd, train_preds_base),4)
precision_base_train = round(precision_score(y_train_pd, train_preds_base),4)
recall_base_train = round(recall_score(y_train_pd, train_preds_base),4)

print(f'F1: {f1_base_train} \nPrecision {precision_base_train} \nRecall: {recall_base_train}')

## Now, we can log this model to [Snowflake Model Registry](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview):

- Log models with important metadata
- Manage model lifecycles
- Serve models from Snowflake runtimes

First, we need to create a Model Registry.

In [ ]:
#Create a snowflake model registry object 
from snowflake.ml.registry import Registry
from snowflake.ml._internal.utils import identifier
from snowflake.ml.model import model_signature

db = identifier._get_unescaped_name(session.get_current_database())
schema = identifier._get_unescaped_name(session.get_current_schema())

# Define model name
model_name = f"MORTGAGE_LENDING_MLOPS_{VERSION_NUM}"

# Create a registry to log the model to
model_registry = Registry(session=session, 
                          database_name=db, 
                          schema_name=schema,
                          options={"enable_monitoring": True})

Now, we can log our model along with all the calculated metrics and other metadata such as model comment (description).

In [ ]:
# Deploy the base model to the model registry
base_version_name = 'XGB_BASE'

try:
    mv_base = model_registry.get_model(model_name).version(base_version_name)
    print("Found existing model version!")
except:
    print("Logging new model version...")
    mv_base = model_registry.log_model(
        model_name=model_name,
        model=xgb_base, 
        version_name=base_version_name,
        sample_input_data = train.drop(["TIMESTAMP", "LOAN_ID", "MORTGAGERESPONSE"]).limit(100), #using snowpark df to maintain lineage
        comment = """ML model for predicting loan approval likelihood.
                    This model was trained using  xgboost classifier.
                    Hyperparameters used were:a
                    max_depth=50, n_estimators=3, learning_rate = 0.75, algorithm = gbtree.
                    """,
        options = {'relax_version': True},
        target_platforms= {"WAREHOUSE"}
    )
    mv_base.set_metric(metric_name="Train_F1_Score", value=f1_base_train)
    mv_base.set_metric(metric_name="Train_Precision_Score", value=precision_base_train)
    mv_base.set_metric(metric_name="Train_Recall_score", value=recall_base_train)

Let's set this base model as our `DEV` model by using model tagging. We'll also create a `PROD` tag to be used later.

In [ ]:
# Create tag for DEV model
session.sql("CREATE OR REPLACE TAG DEV")

In [ ]:
# Create tag for PROD model
session.sql("CREATE OR REPLACE TAG PROD")

In [ ]:
# Apply tag
m = model_registry.get_model(model_name)
m.set_tag("DEV", base_version_name)
m.comment = "Loan approval prediction models" # Set model level comment

The next few cells demonstrate different functions available to us to see our existing models, their versions, metadata such as metrics, and available model functions.

In [ ]:
model_registry.show_models()

In [ ]:
model_registry.get_model(model_name).show_versions()

In [ ]:
print(mv_base)
print(mv_base.show_metrics())

In [ ]:
mv_base.show_functions()

Let's now test running inference on our test set using our logged model in the Registry.

In [ ]:
reg_preds = mv_base.run(test, function_name = "predict")
reg_preds = reg_preds.rename(col('"output_feature_0"'), "MORTGAGE_PREDICTION")
reg_preds.show(10)

In [ ]:
preds_pd = reg_preds.select(["MORTGAGERESPONSE", "MORTGAGE_PREDICTION"]).to_pandas()
f1_base_test = round(f1_score(preds_pd.MORTGAGERESPONSE, preds_pd.MORTGAGE_PREDICTION),4)
precision_base_test = round(precision_score(preds_pd.MORTGAGERESPONSE, preds_pd.MORTGAGE_PREDICTION),4)
recall_base_test = round(recall_score(preds_pd.MORTGAGERESPONSE, preds_pd.MORTGAGE_PREDICTION),4)

# Log metrics to model registry model
mv_base.set_metric(metric_name="Test_F1_Score", value=f1_base_test)
mv_base.set_metric(metric_name="Test_Precision_Score", value=precision_base_test)
mv_base.set_metric(metric_name="Test_Recall_score", value=recall_base_test)

print(f'F1: {f1_base_test} \nPrecision {precision_base_test} \nRecall: {recall_base_test}')

### Oh no! Our model's performance seems to have dropped off significantly from training to our test set. 

### This is evidence that our model is overfit - can we fix this with Snowflake's [Parallel Hyperparameter Optimization](https://docs.snowflake.com/en/developer-guide/snowflake-ml/container-hpo)?

In [ ]:
X_train = train.drop("MORTGAGERESPONSE", "TIMESTAMP", "LOAN_ID")
y_train = train.select("MORTGAGERESPONSE")
X_test = test.drop("MORTGAGERESPONSE","TIMESTAMP", "LOAN_ID")
y_test = test.select("MORTGAGERESPONSE")

We first ingest data from the Snowpark dataframes through the Container Runtime DataConnector API. Then, we define a training function that creates an XGBoost model. The Tuner interface provides the tuning functionality, based on the given training function and search space.

In [ ]:
from snowflake.ml.data import DataConnector
from snowflake.ml.modeling.tune import get_tuner_context
from snowflake.ml.modeling import tune
from entities import search_algorithm

# Define dataset map
dataset_map = {
    "x_train": DataConnector.from_dataframe(X_train),
    "y_train": DataConnector.from_dataframe(y_train),
    "x_test": DataConnector.from_dataframe(X_test),
    "y_test": DataConnector.from_dataframe(y_test)
    }


# Define a training function, with any models you choose within it.
def train_func():
    # A context object provided by HPO API to expose data for the current HPO trial
    tuner_context = get_tuner_context()
    config = tuner_context.get_hyper_params()
    dm = tuner_context.get_dataset_map()

    model = XGBClassifier(**config, random_state=42)
    model.fit(dm["x_train"].to_pandas().sort_index(), dm["y_train"].to_pandas().sort_index())
    f1_metric = f1_score(
        dm["y_train"].to_pandas().sort_index(), model.predict(dm["x_train"].to_pandas().sort_index())
    )
    tuner_context.report(metrics={"f1_score": f1_metric}, model=model)

tuner = tune.Tuner(
    train_func=train_func,
    search_space={
        "max_depth": tune.randint(1, 20),
        "learning_rate": tune.uniform(0.01, 0.1),
        "n_estimators": tune.randint(50, 100),
    },
    tuner_config=tune.TunerConfig(
        metric="f1_score",
        mode="max",
        search_alg=search_algorithm.RandomSearch(random_state=101),
        num_trials=8, #run 8 trial runs
        max_concurrent_trials=4, #use 4 gpus concurrently
    ),
)

Now, we can run the Tuner and get the best model.

In [ ]:
# Snowflake HPO in PrPr is single node, ensure ray context only knows of head node.
tuner_results = tuner.run(dataset_map=dataset_map)

In [ ]:
tuned_model = tuner_results.best_model
tuned_model

Let's compute the train and test metrics.

In [ ]:
#Generate predictions
xgb_opt_preds = tuned_model.predict(train_pd.drop(["TIMESTAMP", "LOAN_ID", "MORTGAGERESPONSE"],axis=1))

#Generate performance metrics
f1_opt_train = round(f1_score(train_pd.MORTGAGERESPONSE, xgb_opt_preds),4)
precision_opt_train = round(precision_score(train_pd.MORTGAGERESPONSE, xgb_opt_preds),4)
recall_opt_train = round(recall_score(train_pd.MORTGAGERESPONSE, xgb_opt_preds),4)

print(f'Train Results: \nF1: {f1_opt_train} \nPrecision {precision_opt_train} \nRecall: {recall_opt_train}')

In [ ]:
#Generate test predictions
xgb_opt_preds_test = tuned_model.predict(test_pd.drop(["TIMESTAMP", "LOAN_ID", "MORTGAGERESPONSE"],axis=1))

#Generate performance metrics on test data
f1_opt_test = round(f1_score(test_pd.MORTGAGERESPONSE, xgb_opt_preds_test),4)
precision_opt_test = round(precision_score(test_pd.MORTGAGERESPONSE, xgb_opt_preds_test),4)
recall_opt_test = round(recall_score(test_pd.MORTGAGERESPONSE, xgb_opt_preds_test),4)

print(f'Test Results: \nF1: {f1_opt_test} \nPrecision {precision_opt_test} \nRecall: {recall_opt_test}')

We see the HPO model has a more modest train accuracy than our base model, but the peformance doesn't drop off during testing.

In [ ]:
#Log the optimized model to the model registry
optimized_version_name = 'XGB_Optimized'

try:
    mv_opt = model_registry.get_model(model_name).version(optimized_version_name)
    print("Found existing model version!")
except:
    print("Logging new model version...")
    mv_opt = model_registry.log_model(
        model_name=model_name,
        model=tuned_model, 
        version_name=optimized_version_name,
        sample_input_data = train.drop(["TIMESTAMP", "LOAN_ID", "MORTGAGERESPONSE"]).limit(100),
        comment = "snow ml model built off feature store using HPO model",
        options = {'relax_version': True},
        target_platforms={"WAREHOUSE"}
    )
    mv_opt.set_metric(metric_name="Train_F1_Score", value=f1_opt_train)
    mv_opt.set_metric(metric_name="Train_Precision_Score", value=precision_opt_train)
    mv_opt.set_metric(metric_name="Train_Recall_score", value=recall_opt_train)

    mv_opt.set_metric(metric_name="Test_F1_Score", value=f1_opt_test)
    mv_opt.set_metric(metric_name="Test_Precision_Score", value=precision_opt_test)
    mv_opt.set_metric(metric_name="Test_Recall_score", value=recall_opt_test)

Let's check what our default model version is in our Model Registry.

In [ ]:
# Here we see the BASE version is our default version
model_registry.get_model(model_name).default

Let's make this optimized model the default version instead and also set it as our `PROD` version using tags.

In [ ]:
# Now we'll set the optimized model to be the default model version going forward
model_registry.get_model(model_name).default = optimized_version_name

In [ ]:
# Now we see our optimized version we have now recently promoted to our DEFAULT model version
model_registry.get_model(model_name).default

In [ ]:
# We'll now update the PROD tagged model to be the optimized model version 
# rather than our overfit base version
m.unset_tag("DEV")
m.set_tag("PROD", optimized_version_name)
m.show_tags()

### Now that we've deployed some model versions and tested inference...

### Let's explain our optimized model!
- ### Snowflake offers [built in explainability capabilities](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/model-explainability) on top of models logged to the model registry
- ### In the below section we'll generate shapley values using these built in functions to understand how input features impact our model's behavior

In [ ]:
# Create a sample of 1000 records
test_pd_sample = test_pd.rename(columns=rename_dict).sample(n=1000, random_state = 100).reset_index(drop=True)

# Compute shapley values for each model
opt_shap_pd = mv_opt.run(test_pd_sample, function_name="explain")

We can take a look at available plots to understand how important each feature is to the prediction for the `opt` model.

In [ ]:
from snowflake.ml.monitoring import explain_visualize

feat_df=test_pd_sample.drop(["MORTGAGERESPONSE","TIMESTAMP", "LOAN_ID"],axis=1)

explain_visualize.plot_influence_sensitivity(opt_shap_pd, feat_df, figsize=(1500, 500))
explain_visualize.plot_force(opt_shap_pd.iloc[0], feat_df.iloc[0], figsize=(1500, 500))

In [ ]:
explain_visualize.plot_violin(opt_shap_pd, feat_df, figsize=(1400, 100))

# Model Observability (Model Monitoring)

[ML Observability](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/model-observability) allows you to track the quality of production models you have deployed via the Snowflake Model Registry across multiple dimensions, such as performance, drift, and volume.

Let's first save our train and test data as Snowflake tables and create a new stage for the next few steps.

In [ ]:
train.write.save_as_table(f"DEMO_MORTGAGE_LENDING_TRAIN_{VERSION_NUM}", mode="overwrite")
test.write.save_as_table(f"DEMO_MORTGAGE_LENDING_TEST_{VERSION_NUM}", mode="overwrite")

In [ ]:
session.sql("CREATE STAGE IF NOT EXISTS ML_STAGE").collect()

We can create stored procedures for running model inference. These procedures can be scheduled or orchestrated to run continuous inference.

In [ ]:
from snowflake import snowpark
from snowflake.ml.registry import Registry
import joblib
import os
import logging
from snowflake.ml.modeling.pipeline import Pipeline
import snowflake.ml.modeling.preprocessing as pp
from snowflake.snowpark.types import StringType, IntegerType
import snowflake.snowpark.functions as F


def demo_inference_sproc(session: snowpark.Session, table_name: str, modelname: str, modelversion: str) -> str:
    database=session.get_current_database()
    schema=session.get_current_schema()
    reg = Registry(session=session)
    m = reg.get_model(model_name)  # Fetch the model using the registry
    mv = m.version(modelversion)
    
    input_table_name=table_name
    pred_col = f'{modelversion}_PREDICTION'

    # Read the input table to a dataframe
    df = session.table(input_table_name)
    # 'results' is the output DataFrame with predictions
    results = mv.run(df, function_name="predict").select("LOAN_ID",'"output_feature_0"').withColumnRenamed('"output_feature_0"', pred_col)

    final = df.join(results, on="LOAN_ID", how="full")
    
    # Write results back to Snowflake table
    final.write.save_as_table(table_name, mode='overwrite',enable_schema_evolution=True)

    return "Success"

# Register the stored procedure
session.sproc.register(
    func=demo_inference_sproc,
    name="model_inference_sproc",
    replace=True,
    is_permanent=True,
    stage_location="@ML_STAGE",
    packages=['joblib', 'snowflake-snowpark-python', 'snowflake-ml-python'],
    return_type=StringType()
)


We'll call our stored procedure 4 times which will execute inference using our 2 models (base and HPO optimized) on our train and test data.

In [ ]:
CALL model_inference_sproc(
    'DEMO_MORTGAGE_LENDING_TRAIN_{{VERSION_NUM}}',
    '{{model_name}}', 
    '{{base_version_name}}');

In [ ]:
CALL model_inference_sproc(
    'DEMO_MORTGAGE_LENDING_TEST_{{VERSION_NUM}}',
    '{{model_name}}', 
    '{{base_version_name}}');

In [ ]:
CALL model_inference_sproc(
    'DEMO_MORTGAGE_LENDING_TRAIN_{{VERSION_NUM}}',
    '{{model_name}}', 
    '{{optimized_version_name}}');

In [ ]:
CALL model_inference_sproc(
    'DEMO_MORTGAGE_LENDING_TEST_{{VERSION_NUM}}',
    '{{model_name}}', 
    '{{optimized_version_name}}');

We can take a look at our predictions now.

In [ ]:
SELECT * FROM DEMO_MORTGAGE_LENDING_TEST_{{VERSION_NUM}} LIMIT 10

### Create Model Monitors

We'll now create model monitors for both the base and HPO optimized models.

In [ ]:
CREATE OR REPLACE MODEL MONITOR MORTGAGE_LENDING_BASE_MODEL_MONITOR
WITH
    MODEL={{model_name}}
    VERSION={{base_version_name}}
    FUNCTION=predict
    SOURCE=DEMO_MORTGAGE_LENDING_TEST_{{VERSION_NUM}}
    BASELINE=DEMO_MORTGAGE_LENDING_TRAIN_{{VERSION_NUM}}
    TIMESTAMP_COLUMN=TIMESTAMP
    PREDICTION_CLASS_COLUMNS=(XGB_BASE_PREDICTION)  
    ACTUAL_CLASS_COLUMNS=(MORTGAGERESPONSE)
    ID_COLUMNS=(LOAN_ID)
    WAREHOUSE={{session.get_current_warehouse()}}
    REFRESH_INTERVAL='1 min'
    AGGREGATION_WINDOW='1 day';

In [ ]:
CREATE OR REPLACE MODEL MONITOR MORTGAGE_LENDING_OPTIMIZED_MODEL_MONITOR
WITH
    MODEL={{model_name}}
    VERSION={{optimized_version_name}}
    FUNCTION=predict
    SOURCE=DEMO_MORTGAGE_LENDING_TEST_{{VERSION_NUM}}
    BASELINE=DEMO_MORTGAGE_LENDING_TRAIN_{{VERSION_NUM}}
    TIMESTAMP_COLUMN=TIMESTAMP
    PREDICTION_CLASS_COLUMNS=(XGB_OPTIMIZED_PREDICTION)  
    ACTUAL_CLASS_COLUMNS=(MORTGAGERESPONSE)
    ID_COLUMNS=(LOAN_ID)
    WAREHOUSE={{session.get_current_warehouse()}}
    REFRESH_INTERVAL='1 min'
    AGGREGATION_WINDOW='1 day';

We can also compute the prediction drift using the Model Monitor.

In [ ]:
SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
'MORTGAGE_LENDING_OPTIMIZED_MODEL_MONITOR', 'DIFFERENCE_OF_MEANS', 'XGB_BASE_PREDICTION', '1 DAY', TO_TIMESTAMP_TZ('2024-06-01'), TO_TIMESTAMP_TZ('2024-09-01')))

## Model Deployment to Snowpark Container Services (SPCS)

Here, we also show how to deploy a model to Snowpark Container Services as a long running service using Model Serving. This is useful to run GPU based inference, run very large models that do not fit in the Warehouse memory, or when you have additional package dependencies that are not met in the Warehouse. 

You can also create REST API endpoints for running inference using external HTTP requests.  Let's first define the variables we'll use to create the deployment service.

In [ ]:
image_repo_name = "my_inference_images"

cp_name = "CPU_X64_M_1"
num_spcs_nodes = '1'
service_name = 'MORTGAGE_LENDING_PREDICTION_SERVICE'

current_database = session.get_current_database().replace('"', '')
current_schema = session.get_current_schema().replace('"', '')
extended_image_repo_name = f"{current_database}.DEFAULT_SCHEMA.{image_repo_name}"
extended_service_name = f'{current_database}.DEFAULT_SCHEMA.{service_name}'

In [ ]:
DROP SERVICE IF EXISTS {{service_name}};

Now we will create a service based on the model we set as the `DEFAULT` and promoted to `prod` to the Model Registry. 

Here we are selecting `ingress_enabled = True` to also create a REST API endpoint for our model.

In [ ]:
model_registry.get_model(model_name).default

In [ ]:
mv_base.create_service(
    service_name=extended_service_name,
    service_compute_pool=cp_name,
    image_repo=extended_image_repo_name,
    ingress_enabled=True,
    max_instances=int(num_spcs_nodes),
    build_external_access_integration="ALLOW_ALL_INTEGRATION"
)

Now, we can check that the service is created.

In [ ]:
SHOW SERVICES;

Now, we can run inference on test data using this deployed SPCS Model Service.

In [ ]:
mv_base.run(test, 
            function_name = "PREDICT", 
            service_name = "DEFAULT_SCHEMA.MORTGAGE_LENDING_PREDICTION_SERVICE")

 Since we created a REST API above, this service will run continuously. It is a good idea to drop or suspend the service if you do not need it. Compute pool will automatically suspend if no service is running.

In [ ]:
DROP SERVICE IF EXISTS {{service_name}};

# Conclusion

#### 🛠️ Snowflake Feature Store tracks feature definitions and maintains lineage of sources and destinations 🛠️
#### 🚀 Snowflake Model Registry gives users a secure and flexible framework to deploy track and monitor models 🚀
#### 🔮 All model versions logged in the Model Registry can be accessed for inference, explainability, lineage tracking, visibility, and deployed 🔮
